In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
import time

print("Libraries loaded successfully!")

In [ ]:
!wget -q http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/facades.tar.gz
!tar -xzf facades.tar.gz

print("Dataset downloaded successfully!")


In [ ]:
!ls facades

In [ ]:
IMG_HEIGHT = 256
IMG_WIDTH = 256
OUTPUT_CHANNELS = 3

BUFFER_SIZE = 400
BATCH_SIZE = 1

In [ ]:
def load_image(image_file):
    image = tf.io.read_file(image_file)
    image = tf.io.decode_jpeg(image)

    image = tf.cast(image, tf.float32)

    width = tf.shape(image)[1]

    input_image = image[:, :width // 2, :]
    real_image = image[:, width // 2:, :]

    return input_image, real_image

In [ ]:
def resize_images(input_image, real_image):
    input_image = tf.image.resize(
        input_image,
        [IMG_HEIGHT, IMG_WIDTH]
    )

    real_image = tf.image.resize(
        real_image,
        [IMG_HEIGHT, IMG_WIDTH]
    )

    return input_image, real_image

In [ ]:
def normalize(input_image, real_image):
    input_image = (input_image / 127.5) - 1
    real_image = (real_image / 127.5) - 1

    return input_image, real_image

In [ ]:
def random_jitter(input_image, real_image):

    input_image = tf.image.resize(
        input_image,
        [286, 286],
        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
    )

    real_image = tf.image.resize(
        real_image,
        [286, 286],
        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
    )

    stacked_image = tf.stack([input_image, real_image], axis=0)

    cropped_image = tf.image.random_crop(
        stacked_image,
        size=[2, IMG_HEIGHT, IMG_WIDTH, 3]
    )

    input_image = cropped_image[0]
    real_image = cropped_image[1]

    if tf.random.uniform(()) > 0.5:
        input_image = tf.image.flip_left_right(input_image)
        real_image = tf.image.flip_left_right(real_image)

    return input_image, real_image

In [ ]:
def load_train_image(image_file):
    input_image, real_image = load_image(image_file)

    input_image, real_image = resize_images(
        input_image,
        real_image
    )

    input_image, real_image = random_jitter(
        input_image,
        real_image
    )

    input_image, real_image = normalize(
        input_image,
        real_image
    )

    return input_image, real_image

In [ ]:
def load_test_image(image_file):
    input_image, real_image = load_image(image_file)

    input_image, real_image = resize_images(
        input_image,
        real_image
    )

    input_image, real_image = normalize(
        input_image,
        real_image
    )

    return input_image, real_image

In [ ]:
train_dataset = tf.data.Dataset.list_files(
    "facades/train/*.jpg"
)

train_dataset = train_dataset.map(
    load_train_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.shuffle(BUFFER_SIZE)
train_dataset = train_dataset.batch(BATCH_SIZE)

In [ ]:
test_dataset = tf.data.Dataset.list_files(
    "facades/test/*.jpg"
)

test_dataset = test_dataset.map(
    load_test_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset.batch(BATCH_SIZE)

In [ ]:
sample_input, sample_real = next(iter(train_dataset))

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title("Input Image")
plt.imshow((sample_input[0] + 1) / 2)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Target Image")
plt.imshow((sample_real[0] + 1) / 2)
plt.axis("off")

plt.show()

In [ ]:
def downsample(filters, size, apply_batchnorm=True):

    initializer = tf.random_normal_initializer(0., 0.02)

    result = tf.keras.Sequential()

    result.add(
        tf.keras.layers.Conv2D(
            filters,
            size,
            strides=2,
            padding='same',
            kernel_initializer=initializer,
            use_bias=not apply_batchnorm
        )
    )

    if apply_batchnorm:
        result.add(tf.keras.layers.BatchNormalization())

    result.add(tf.keras.layers.LeakyReLU())

    return result

In [ ]:
def upsample(filters, size, apply_dropout=False):

    initializer = tf.random_normal_initializer(0., 0.02)

    result = tf.keras.Sequential()

    result.add(
        tf.keras.layers.Conv2DTranspose(
            filters,
            size,
            strides=2,
            padding='same',
            kernel_initializer=initializer,
            use_bias=False
        )
    )

    result.add(tf.keras.layers.BatchNormalization())

    if apply_dropout:
        result.add(tf.keras.layers.Dropout(0.5))

    result.add(tf.keras.layers.ReLU())

    return result

In [ ]:
def Generator():

    inputs = tf.keras.layers.Input(
        shape=[256, 256, 3]
    )

    down_stack = [
        downsample(64, 4, apply_batchnorm=False),
        downsample(128, 4),
        downsample(256, 4),
        downsample(512, 4),
        downsample(512, 4),
        downsample(512, 4),
        downsample(512, 4),
        downsample(512, 4),
    ]

    up_stack = [
        upsample(512, 4, apply_dropout=True),
        upsample(512, 4, apply_dropout=True),
        upsample(512, 4, apply_dropout=True),
        upsample(512, 4),
        upsample(256, 4),
        upsample(128, 4),
        upsample(64, 4),
    ]

    initializer = tf.random_normal_initializer(0., 0.02)

    last = tf.keras.layers.Conv2DTranspose(
        3,
        4,
        strides=2,
        padding='same',
        kernel_initializer=initializer,
        activation='tanh'
    )

    x = inputs

    skips = []

    for down in down_stack:
        x = down(x)
        skips.append(x)

    skips = reversed(skips[:-1])

    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = tf.keras.layers.Concatenate()([x, skip])

    x = last(x)

    return tf.keras.Model(
        inputs=inputs,
        outputs=x
    )

In [ ]:
generator = Generator()

generator.summary()

In [ ]:
generated_image = generator(sample_input)

print("Generated image shape:")
print(generated_image.shape)

In [ ]:
def Discriminator():

    initializer = tf.random_normal_initializer(0., 0.02)

    inp = tf.keras.layers.Input(
        shape=[256, 256, 3],
        name='input_image'
    )

    tar = tf.keras.layers.Input(
        shape=[256, 256, 3],
        name='target_image'
    )

    x = tf.keras.layers.concatenate([inp, tar])

    down1 = downsample(
        64,
        4,
        False
    )(x)

    down2 = downsample(
        128,
        4
    )(down1)

    down3 = downsample(
        256,
        4
    )(down2)

    zero_pad1 = tf.keras.layers.ZeroPadding2D()(down3)

    conv = tf.keras.layers.Conv2D(
        512,
        4,
        strides=1,
        kernel_initializer=initializer,
        use_bias=False
    )(zero_pad1)

    batchnorm1 = tf.keras.layers.BatchNormalization()(conv)

    leaky_relu = tf.keras.layers.LeakyReLU()(batchnorm1)

    zero_pad2 = tf.keras.layers.ZeroPadding2D()(leaky_relu)

    last = tf.keras.layers.Conv2D(
        1,
        4,
        strides=1,
        kernel_initializer=initializer
    )(zero_pad2)

    return tf.keras.Model(
        inputs=[inp, tar],
        outputs=last
    )

In [ ]:
discriminator = Discriminator()

discriminator.summary()

In [ ]:
discriminator_output = discriminator(
    [sample_input, sample_real]
)

print(discriminator_output.shape)

In [ ]:
loss_object = tf.keras.losses.BinaryCrossentropy(
    from_logits=True
)

In [ ]:
def generator_loss(
    disc_generated_output,
    gen_output,
    target
):

    gan_loss = loss_object(
        tf.ones_like(disc_generated_output),
        disc_generated_output
    )

    l1_loss = tf.reduce_mean(
        tf.abs(target - gen_output)
    )

    total_gen_loss = gan_loss + (100 * l1_loss)

    return total_gen_loss

In [ ]:
def discriminator_loss(
    disc_real_output,
    disc_generated_output
):

    real_loss = loss_object(
        tf.ones_like(disc_real_output),
        disc_real_output
    )

    generated_loss = loss_object(
        tf.zeros_like(disc_generated_output),
        disc_generated_output
    )

    total_disc_loss = real_loss + generated_loss

    return total_disc_loss

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(
    2e-4,
    beta_1=0.5
)

discriminator_optimizer = tf.keras.optimizers.Adam(
    2e-4,
    beta_1=0.5
)

In [ ]:
@tf.function
def train_step(input_image, target):

    with tf.GradientTape() as gen_tape, \
         tf.GradientTape() as disc_tape:

        gen_output = generator(
            input_image,
            training=True
        )

        disc_real_output = discriminator(
            [input_image, target],
            training=True
        )

        disc_generated_output = discriminator(
            [input_image, gen_output],
            training=True
        )

        gen_loss = generator_loss(
            disc_generated_output,
            gen_output,
            target
        )

        disc_loss = discriminator_loss(
            disc_real_output,
            disc_generated_output
        )

    generator_gradients = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    discriminator_gradients = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(
            generator_gradients,
            generator.trainable_variables
        )
    )

    discriminator_optimizer.apply_gradients(
        zip(
            discriminator_gradients,
            discriminator.trainable_variables
        )
    )

    return gen_loss, disc_loss

In [ ]:
def generate_images(model, test_input, tar):

    prediction = model(test_input, training=False)

    plt.figure(figsize=(15, 5))

    display_list = [
        test_input[0],
        tar[0],
        prediction[0]
    ]

    title = [
        'Input Image',
        'Target Image',
        'Generated Image'
    ]

    for i in range(3):

        plt.subplot(1, 3, i + 1)

        plt.title(title[i])

        plt.imshow(
            (display_list[i] + 1) / 2
        )

        plt.axis('off')

    plt.show()

In [ ]:
for example_input, example_target in test_dataset.take(1):

    generate_images(
        generator,
        example_input,
        example_target
    )

In [ ]:
EPOCHS = 20

for epoch in range(EPOCHS):

    start = time.time()

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    for input_image, target in train_dataset:

        gen_loss, disc_loss = train_step(
            input_image,
            target
        )

    print("Generator Loss:", float(gen_loss))
    print("Discriminator Loss:", float(disc_loss))

    for example_input, example_target in test_dataset.take(1):

        generate_images(
            generator,
            example_input,
            example_target
        )

    print(
        "Time taken:",
        round(time.time() - start, 2),
        "seconds"
    )

In [ ]:
os.makedirs("saved_model", exist_ok=True)

In [ ]:
generator.save(
    "saved_model/pix2pix_generator.keras"
)

In [ ]:
discriminator.save(
    "saved_model/pix2pix_discriminator.keras"
)

In [ ]:
from google.colab import files

files.download(
    "saved_model/pix2pix_generator.keras"
)